# SplitComp: Parameter Sweep

Three-action Becker compliance model (comply, evade, split) with six parameters in [0, 1]. We map where compliance breaks and which policy levers matter most.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats.qmc import LatinHypercube

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.size': 8,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.titlesize': 9,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
})

# Baseline: weak enforcement, low coordination, moderate benefit
BASELINE = {
    'B': 0.60,
    'a': 0.40,
    'd': 0.50,
    'F': 0.50,
    'c': 0.10,
    'sigma': 0.25,
}

PARAMS = list(BASELINE.keys())

# Blue, vermillion, amber (colorblind-safe)
ACTION_COLORS = ['#2166AC', '#B2182B', '#E8A838']
ACTION_NAMES = ['Comply', 'Evade', 'Split']
CMAP = ListedColormap(ACTION_COLORS)
NORM = BoundaryNorm([-.5, .5, 1.5, 2.5], CMAP.N)

PARAM_LABELS = {
    'B': r'Benefit $B$', 'a': r'Audit prob $a$',
    'd': r'Detection $d$', 'F': r'Fine $F$',
    'c': r'Split cost $c$', 'sigma': r'Coordination $\sigma$',
}

## Decision Function

The lab picks whichever action has the highest payoff:

$$U(\text{comply}) = 0, \quad U(\text{evade}) = B - adF, \quad U(\text{split}) = B - c - ad\sigma F$$

In [ ]:
def lab_decision(B, a, d, F, c, sigma):
    """Return 0=comply, 1=evade, 2=split based on argmax payoff.
    Ties go to comply (lowest index)."""
    u_comply = np.zeros_like(B, dtype=float)
    u_evade = B - a * d * F
    u_split = B - c - a * d * sigma * F
    payoffs = np.stack([u_comply, u_evade, u_split], axis=-1)
    return np.argmax(payoffs, axis=-1)

## Analytic Decision Boundaries

Three closed-form boundaries partition the parameter space. These get overlaid on the heatmaps.

- **Comply = Evade:** $B = adF$
- **Comply = Split:** $B = c + ad\sigma F$
- **Evade = Split:** $c = adF(1 - \sigma)$

In [ ]:
def _solve_boundary(x_vals, x_param, y_param, held, equation):
    """Solve a boundary equation for y_param over a sweep of x_param.
    Returns only points where the solution falls in [0, 1]."""
    results = np.full_like(x_vals, np.nan)
    for i, xv in enumerate(x_vals):
        p = dict(held)
        p[x_param] = xv
        try:
            results[i] = equation(p)
        except (ZeroDivisionError, FloatingPointError):
            pass
    mask = (results >= 0) & (results <= 1)
    return x_vals[mask], results[mask]


# Each boundary function rearranges its equation to isolate y_param.

def boundary_comply_evade(x_vals, x_param, y_param, held):
    """B = a*d*F"""
    def eq(p):
        adf = p.get('a', BASELINE['a']) * p.get('d', BASELINE['d']) * p.get('F', BASELINE['F'])
        if y_param == 'B':
            return adf
        elif y_param in ('a', 'd', 'F'):
            others = {'a', 'd', 'F'} - {y_param}
            denom = 1.0
            for o in others:
                denom *= p.get(o, BASELINE[o])
            return p.get('B', BASELINE['B']) / denom if denom else np.nan
        return np.nan
    return _solve_boundary(x_vals, x_param, y_param, held, eq)


def boundary_comply_split(x_vals, x_param, y_param, held):
    """B = c + a*d*sigma*F"""
    def eq(p):
        a = p.get('a', BASELINE['a'])
        d = p.get('d', BASELINE['d'])
        F = p.get('F', BASELINE['F'])
        c = p.get('c', BASELINE['c'])
        sigma = p.get('sigma', BASELINE['sigma'])
        if y_param == 'B':
            return c + a * d * sigma * F
        elif y_param == 'sigma':
            denom = a * d * F
            return (p.get('B', BASELINE['B']) - c) / denom if denom else np.nan
        elif y_param == 'c':
            return p.get('B', BASELINE['B']) - a * d * sigma * F
        return np.nan
    return _solve_boundary(x_vals, x_param, y_param, held, eq)


def boundary_evade_split(x_vals, x_param, y_param, held):
    """c = a*d*F*(1-sigma)"""
    def eq(p):
        a = p.get('a', BASELINE['a'])
        d = p.get('d', BASELINE['d'])
        F = p.get('F', BASELINE['F'])
        c = p.get('c', BASELINE['c'])
        sigma = p.get('sigma', BASELINE['sigma'])
        if y_param == 'sigma':
            denom = a * d * F
            return 1.0 - c / denom if denom else np.nan
        elif y_param == 'c':
            return a * d * F * (1.0 - sigma)
        return np.nan
    return _solve_boundary(x_vals, x_param, y_param, held, eq)

## 2D Heatmaps

Sweep two parameters from 0 to 1, hold the rest at baseline. Blue = comply, vermillion = evade, amber = split.

In [ ]:
def plot_heatmap(x_param, y_param, ax, overrides=None, title=None, res=200):
    """Sweep x_param and y_param, color each cell by optimal action."""
    xs = np.linspace(0.01, 1, res)
    ys = np.linspace(0.01, 1, res)
    X, Y = np.meshgrid(xs, ys)

    # Start from baseline, then apply any overrides and the swept axes
    params = {k: np.full_like(X, v) for k, v in BASELINE.items()}
    if overrides:
        for k, v in overrides.items():
            params[k] = np.full_like(X, v)
    params[x_param] = X
    params[y_param] = Y

    Z = lab_decision(**params)

    ax.imshow(Z, origin='lower', extent=[0.01, 1, 0.01, 1],
              aspect='auto', cmap=CMAP, norm=NORM, interpolation='nearest')

    # Overlay analytic boundary lines
    held = {k: (overrides or {}).get(k, BASELINE[k])
            for k in PARAMS if k not in (x_param, y_param)}

    for bfunc, color, ls in [(boundary_comply_evade, 'white', '-'),
                              (boundary_comply_split, 'black', '-'),
                              (boundary_evade_split, 'black', '--')]:
        bx, by = bfunc(xs, x_param, y_param, held)
        if len(bx) > 1:
            ax.plot(bx, by, color=color, ls=ls, lw=1.2)

    ax.set_xlabel(PARAM_LABELS[x_param])
    ax.set_ylabel(PARAM_LABELS[y_param])
    if title:
        ax.set_title(title)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6))

plot_heatmap('a', 'B', axes[0, 0], title=r'Benefit vs Audit Prob')
plot_heatmap('F', 'B', axes[0, 1], title=r'Benefit vs Fine')
plot_heatmap('sigma', 'c', axes[1, 0], title=r'Split Cost vs Coordination')
plot_heatmap('d', 'a', axes[1, 1], title=r'Audit Prob vs Detection')

handles = [plt.Rectangle((0, 0), 1, 1, fc=c) for c in ACTION_COLORS]
fig.legend(handles, ACTION_NAMES, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False)

fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig('figures/heatmaps.png', bbox_inches='tight')
plt.show()

## B vs. Coordination Under Strong Enforcement

At baseline enforcement ($adF = 0.10$), the penalty reduction from splitting just equals $c = 0.10$, so splitting never wins. Raising to $a = d = F = 0.6$ ($adF = 0.216$) reveals all three regions. The split wedge disappears at $\sigma = 1 - c/(adF) \approx 0.54$.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

strong = {'a': 0.60, 'd': 0.60, 'F': 0.60}
plot_heatmap('sigma', 'B', ax, overrides=strong,
             title=r'Benefit vs Coordination ($a = d = F = 0.6$)')

handles = [plt.Rectangle((0, 0), 1, 1, fc=c) for c in ACTION_COLORS]
fig.legend(handles, ACTION_NAMES, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False)

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig('figures/heatmap_B_vs_sigma.png', bbox_inches='tight')
plt.show()

## Latin Hypercube Sampling

Sample all six parameters simultaneously to estimate what fraction of the full parameter space favors each action.

In [ ]:
N_LHS = 50_000

sampler = LatinHypercube(d=6, seed=42)
samples = sampler.random(n=N_LHS)

decisions = lab_decision(
    B=samples[:, 0], a=samples[:, 1], d=samples[:, 2],
    F=samples[:, 3], c=samples[:, 4], sigma=samples[:, 5],
)

fractions = {name: np.mean(decisions == i) for i, name in enumerate(ACTION_NAMES)}
for name, frac in fractions.items():
    print(f'{name}: {frac:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))

bars = ax.bar(ACTION_NAMES, fractions.values(), color=ACTION_COLORS, edgecolor='black', lw=0.5)
for bar, frac in zip(bars, fractions.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{frac:.1%}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Fraction of parameter space')
ax.set_title('LHS Action Distribution (n=50,000)')
ax.set_ylim(0, 1)

fig.tight_layout()
fig.savefig('figures/lhs_distribution.png', bbox_inches='tight')
plt.show()

## Sensitivity Analysis

Hold each parameter at 0.2 and 0.8, sample the other five with LHS, and measure the compliance rate swing.

In [ ]:
N_SENS = 10_000

def compliance_rate_fixed(param_idx, fixed_val, n=N_SENS):
    """Hold one parameter constant, sample the other five, return compliance fraction."""
    sampler = LatinHypercube(d=5, seed=param_idx * 1000 + int(fixed_val * 100))
    other_samples = sampler.random(n=n)
    # Insert the fixed column back at its original position
    full = np.insert(other_samples, param_idx, fixed_val, axis=1)
    decisions = lab_decision(
        B=full[:, 0], a=full[:, 1], d=full[:, 2],
        F=full[:, 3], c=full[:, 4], sigma=full[:, 5],
    )
    return np.mean(decisions == 0)

swings = {}
for idx, param in enumerate(PARAMS):
    rate_low = compliance_rate_fixed(idx, 0.2)
    rate_high = compliance_rate_fixed(idx, 0.8)
    swings[param] = (rate_low, rate_high)

for param, (lo, hi) in swings.items():
    print(f'{param}: low={lo:.1%}, high={hi:.1%}, swing={hi - lo:+.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))

sorted_params = sorted(swings.keys(), key=lambda p: abs(swings[p][1] - swings[p][0]))
y_pos = np.arange(len(sorted_params))

for i, param in enumerate(sorted_params):
    lo, hi = swings[param]
    ax.barh(i, hi - lo, left=lo, color='#2166AC' if hi > lo else '#B2182B',
            edgecolor='black', lw=0.5, height=0.6)
    ax.plot([lo, lo], [i - 0.3, i + 0.3], color='black', lw=1)
    ax.plot([hi, hi], [i - 0.3, i + 0.3], color='black', lw=1)

ax.set_yticks(y_pos)
ax.set_yticklabels([PARAM_LABELS[p] for p in sorted_params])
ax.set_xlabel('Compliance rate')
ax.set_title('Sensitivity Tornado: Compliance Rate Swing')
ax.axvline(fractions['Comply'], color='gray', ls=':', lw=0.8)

fig.tight_layout()
fig.savefig('figures/sensitivity_tornado.png', bbox_inches='tight')
plt.show()